In [ ]:
!pip install kaggle
!kaggle datasets download -d fronkongames/steam-games-dataset
!unzip steam-games-dataset.zip
!rm steam-games-dataset.zip

In [ ]:
# for app in dataset:
#   appID = app                                         # AppID, unique identifier for each app (string).
#   game = dataset[app]

#   name = game['name']                                 # Game name (string).
#   releaseDate = game['release_date']                  # Release date (string).
#   estimatedOwners = game['estimated_owners']          # Estimated owners (string, e.g.: "0 - 20000").
#   peakCCU = game['peak_ccu']                          # Number of concurrent users, yesterday (int).
#   required_age = game['required_age']                 # Age required to play, 0 if it is for all audiences (int).
#   price = game['price']                               # Price in USD, 0.0 if its free (float).
#   dlcCount = game['dlc_count']                        # Number of DLCs, 0 if you have none (int).
#   longDesc = game['detailed_description']             # Detailed description of the game (string).
#   shortDesc = game['short_description']               # Brief description of the game,
#                                                       # does not contain HTML tags (string).
#   languages = game['supported_languages']             # Comma-separated enumeration of supporting languages.
#   fullAudioLanguages = game['full_audio_languages']   # Comma-separated enumeration of languages with audio support.
#   reviews = game['reviews']                           #
#   headerImage = game['header_image']                  # Header image URL in the store (string).
#   website = game['website']                           # Game website (string).
#   supportWeb = game['support_url']                    # Game support URL (string).
#   supportEmail = game['support_email']                # Game support email (string).
#   supportWindows = game['windows']                    # Does it support Windows? (bool).
#   supportMac = game['mac']                            # Does it support Mac? (bool).
#   supportLinux = game['linux']                        # Does it support Linux? (bool).
#   metacriticScore = game['metacritic_score']          # Metacritic score, 0 if it has none (int).
#   metacriticURL = game['metacritic_url']              # Metacritic review URL (string).
#   userScore = game['user_score']                      # Users score, 0 if it has none (int).
#   positive = game['positive']                         # Positive votes (int).
#   negative = game['negative']                         # Negative votes (int).
#   scoreRank = game['score_rank']                      # Score rank of the game based on user reviews (string).
#   achievements = game['achievements']                 # Number of achievements, 0 if it has none (int).
#   recommens = game['recommendations']                 # User recommendations, 0 if it has none (int).
#   notes = game['notes']                               # Extra information about the game content (string).
#   averagePlaytime = game['average_playtime_forever']  # Average playtime since March 2009, in minutes (int).
#   averageplaytime2W = game['average_playtime_2weeks'] # Average playtime in the last two weeks, in minutes (int).
#   medianPlaytime = game['median_playtime_forever']    # Median playtime since March 2009, in minutes (int).
#   medianPlaytime2W = game['median_playtime_2weeks']   # Median playtime in the last two weeks, in minutes (int).

#   packages = game['packages']                         # Available packages.
#   for pack in packages:
#     title = pack['title']                             # Package title (string).
#     packDesc = pack['description']                    # Package description (string).

#     subs = pack['subs']                               # Subpackages.
#     for sub in subs:
#       text = sub['text']                              # Subpackage title (string).
#       subDesc = sub['description']                    # Subpackage description (string).
#       subPrice = sub['price']                         # Subpackage price in USD (float).

#   developers = game['developers']                     # Game developers.
#   for developer in developers:
#     developerName = developer                         # Developer name (string).

#   publishers = game['publishers']                     # Game publishers.
#   for publisher in publishers:
#     publisherName = publisher                         # Publisher name (string).

#   categories = game['categories']                     # Game categories.
#   for category in categories:
#     categoryName = category                           # Category name (string).

#   genres = game['genres']                             # Game genres.
#   for gender in genres:
#     genderName = gender                               # Gender name (string).

#   screenshots = game['scrennshots']                   # Game screenshots.
#   for screenshot in screenshots:
#     scrennshotsURL = screenshot                       # Game screenshot URL (string).

#   movies = game['movies']                             # Game movies.
#   for movie in movies:
#     movieURL = movie                                  # Game movie URL (string).

#   tags = game['tags']                                 # Tags.
#   for tag in tags:
#     tagKey = tag                                      # Tag key (string, int).

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.DataFrame()
df = pd.read_csv('games.csv')

In [ ]:
df.info()
df.head()

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
  results = np.zeros((len(sequences), dimension))
  for i, sequence in enumerate(sequences):
    for j in sequence:
      results[i, j] = 1.
  return results

In [ ]:
# Step 2: Prepare Data (Extract relevant fields)
game_data = []
user_game_interactions = []  # For collaborative filtering

for app in dataset:
    game = dataset[app]

    game_id = app #Unique String ID
    name = game['name']
    price = game['price']
    average_playtime = game.get('average_playtime_forever', 0)
    user_score = game.get('user_score', 0)
    genres = game.get('genres', [])
    categories = game.get('categories', [])

    # Game metadata features (content-based)
    genres = [genre.get('description', '') for genre in game.get('genres', []) if isinstance(genre, dict)]
    categories = [category.get('description', '') for category in game.get('categories', []) if isinstance(category, dict)]

    # For collaborative filtering, we assume interactions exist (can be user-played)
    for user_id in range(1, 11):  # Example: Simulate 10 users
        user_game_interactions.append([user_id, game_id, np.random.randint(1, 6)])  # Random interaction score (1-5)

    game_data.append({
        'game_id': game_id,
        'name': name,
        'price': price,
        'average_playtime': average_playtime,
        'user_score': user_score,
        'genres': genres,
        'categories': categories
    })

game_data = pd.DataFrame(game_data)
game_data.info()

# !!*Currently Here Working on Multi-Hot encoding*!!

In [ ]:
# Step 3: Prepare the content-based filtering features (e.g., genres, categories)
# Flatten the genres and categories into a one-hot encoding
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()

for app in dataset:
  game = dataset[app]
  genres = game.get('genres', [])
  game['genres_encoded'] = mlb.fit(genres)
  game['categories_encoded'] = mlb.fit(game['categories'])

class_names = mlb.classes_
print(class_names)




In [ ]:
for genre in unique_genres:
  print(genre)

for category in unique_categories:
  print(category)

In [ ]:
# Step 4: Prepare the collaborative filtering matrix (user-game interaction)


In [ ]:
# Step 5: Define the Collaborative Filtering Model (using embeddings)
num_users = user_game_interactions.shape[0]
num_games = len(dataset)

user_input = Input(shape=(1,))
user_embedding = Embedding(input_dim=num_users, output_dim=10)(user_input)
user_embedding = Flatten()(user_embedding)

game_input = Input(shape=(1,))
game_embedding = Embedding(input_dim=num_games, output_dim=10)(game_input)
game_embedding = Flatten()(game_embedding)

collab_output = Concatenate()([user_embedding, game_embedding])
collab_output = Dense(64, activation='relu')(collab_output)
collab_output = Dense(1, activation='linear')(collab_output)

In [ ]:
# Step 6: Define the Content-Based Filtering Model (game metadata)
genre_input = Input(shape=(1,))
genre_embedding = Embedding(input_dim=len(unique_genres), output_dim=5)(genre_input)
genre_embedding = Flatten()(genre_embedding)

category_input = Input(shape=(1,))
category_embedding = Embedding(input_dim=len(unique_categories), output_dim=5)(category_input)
category_embedding = Flatten()(category_embedding)

price_input = Input(shape=(1,))
price_dense = Dense(64, activation='relu')(price_input)

metacritic_input = Input(shape=(1,))
metacritic_dense = Dense(64, activation='relu')(metacritic_input)

content_output = Concatenate()([genre_embedding, category_embedding, price_dense, metacritic_dense])
content_output = Dense(64, activation='relu')(content_output)
content_output = Dense(1, activation='linear')(content_output)

In [ ]:
# Step 7: Hybrid Model (Combining Collaborative and Content-Based Outputs)
final_output = Concatenate()([collab_output, content_output])
final_output = Dense(64, activation='relu')(final_output)
final_output = Dense(1, activation='linear')(final_output)

# Create the model
model = Model(inputs=[user_input, game_input, genre_input, category_input, price_input, metacritic_input], outputs=final_output)

# Compile the model
model.compile(optimizer='adam', loss='mse')

In [ ]:
# Step 8: Prepare Data for Training
# Example user interaction data (training the hybrid model)
X_collab = [user_game_interactions.index.values,
             # Convert game IDs to numeric using factorize
             pd.factorize(user_game_interactions.columns)[0]]
X_content = [np.array([game['genres_encoded'][0] if game['genres_encoded'] else 0 for game in game_data]),
             np.array([game['categories_encoded'][0] if game['categories_encoded'] else 0 for game in game_data]),
             np.array([game['price'] if game['price'] is not None else 0 for game in game_data]),
             np.array([game['user_score'] for game in game_data])]

# Simulate random target values (could be interaction or user preferences)
# Check if X_collab[0] is empty before generating y_train
if len(X_collab[0]) > 0:
    y_train = np.random.rand(len(X_collab[0]))
else:
    # Handle the case where X_collab[0] is empty (e.g., no user interactions)
    # You might want to skip training or use a different approach
    print("No user interactions found. Skipping training.")
    y_train = np.array([])  # Create an empty array to avoid errors

In [ ]:
# Step 9: Train the Model
# Only train if y_train is not empty
if len(y_train) > 0:
    model.fit(X_collab + X_content, y_train, epochs=10, batch_size=64)
else:
    print("Skipping model training due to empty training data.")
# Step 10: Save the Model
model.save('steam_game_recommender.h5')

print("Model training complete!")
